# Data cleaning for anomalies: to long , to short

In [1]:
import pandas as pd
import pyodbc  
import matplotlib.pyplot as plt

## Connection with db

In [2]:
conn = pyodbc.connect(r'DRIVER={ODBC Driver 17 for SQL Server};SERVER=;DATABASE=CRH;Trusted_Connection=yes;')
cursor = conn.cursor()

## Data in dataframe

In [3]:
query =  """
        SELECT c.ID, c.InstanceId, TestStartTime, TestFinishTime, Gender, Qualification
        FROM CandidateResultFCA cf 
        JOIN Candidate c
        ON cf.CandidateID = c.ID and cf.InstanceID = c.InstanceID
        """
df = pd.read_sql_query(query, conn)

C:\Users\Lorem\AppData\Local\Temp\ipykernel_17836\970591828.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


In [ ]:
fca_csv = "..\\files\\csv\\fca_investigate.csv" # TODO: outdated 
fca_df = pd.read_csv(fca_csv)

In [5]:
fca_df

,InstrumentClassId,CandidateId,InstanceId,ItemId,Answer1,Answer2,Answer3,CorrectAnswers,TimeSpent
0,1137,3164626,1,407,3,4,5,"2, 3, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0",116
1,1137,3164626,1,410,0,0,0,"2, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0",191
2,1137,3164626,1,411,4,1,5,"5, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0",121
3,1137,3164626,1,412,3,5,4,"1, 2, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0",45
4,1137,3164626,1,413,5,3,1,"1, 5, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0",75
5,1137,3164626,1,414,4,3,5,"1, 5, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0",75
6,1137,3164626,1,415,5,4,3,"5, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0",112
7,1137,3164626,1,416,5,3,4,"3, 1, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0",68
8,1137,3164626,1,417,5,3,4,"4, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0",66
9,1137,3164626,1,418,5,3,4,"4, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0",1268


In [6]:
df

,ID,TestStartTime,TestFinishTime,Gender,Qualification
0,992,2023-09-08 07:07:29.533,2023-09-08 07:09:26.197,Gender_Male,Qualification_Unknown
1,1124,2023-09-27 16:03:59.047,2023-09-27 16:46:19.430,Gender_Female,Qualification_Unknown
2,1158,2023-09-29 14:43:52.897,2023-09-29 15:04:57.667,Gender_Male,Qualification_Bachelor
3,1285,2023-10-13 09:23:15.460,2023-10-13 09:52:39.973,Gender_Male,Qualification_Unknown
4,1485,2023-10-23 07:47:29.690,2023-10-23 08:19:23.397,Gender_Male,Qualification_Bachelor
...,...,...,...,...,...
160213,4982046,2024-02-19 08:34:39.187,2024-02-19 09:04:19.400,Gender_Male,Qualification_Unknown
160214,4982097,2024-02-08 10:58:04.770,2024-02-08 11:35:20.650,Gender_Male,Qualification_Unknown
160215,4982163,2024-02-22 12:34:33.257,2024-02-22 13:26:31.967,Gender_Male,Qualification_Unknown
160216,4982214,2024-02-01 12:47:48.603,2024-02-01 13:20:29.923,Gender_Male,Qualification_Unknown


In [6]:
fca_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   InstrumentClassId  56 non-null     int64 
 1   CandidateId        56 non-null     int64 
 2   InstanceId         56 non-null     int64 
 3   ItemId             56 non-null     int64 
 4   Answer1            56 non-null     int64 
 5   Answer2            56 non-null     int64 
 6   Answer3            56 non-null     int64 
 7   CorrectAnswers     56 non-null     object
 8   TimeSpent          56 non-null     int64 
dtypes: int64(8), object(1)
memory usage: 4.1+ KB


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160218 entries, 0 to 160217
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   ID              160218 non-null  int64         
 1   TestStartTime   160218 non-null  datetime64[ns]
 2   TestFinishTime  160218 non-null  datetime64[ns]
 3   Gender          160218 non-null  object        
 4   Qualification   160218 non-null  object        
dtypes: datetime64[ns](2), int64(1), object(2)
memory usage: 6.1+ MB


In [8]:
df['Gender'].value_counts()

Gender
Gender_Female     75593
Gender_Male       73978
Gender_Unknown    10647
Name: count, dtype: int64

## Data cleaning

In [9]:
df['Gender'] = df['Gender'].replace({
    'Gender_Female': 'female',
    'Gender_Male': 'male',
    'Gender_Unknown': 'other'
})

In [10]:
df['Qualification'].value_counts()

Qualification
Qualification_Unknown         125401
Qualification_Bachelor         13059
Qualification_Master           12775
Qualification_Secondary         5864
Qualification_PostGraduate      1267
Qualification_Professional       909
Qualification_MBA                488
Qualification_Vocational         370
Qualification_PHD                 84
                                   1
Name: count, dtype: int64

In [11]:
df['Qualification'] = df['Qualification'].replace({
    'Qualification_Unknown': 'unknown',
    'Qualification_Bachelor': 'bachelor',
    'Qualification_Master': 'master',
    'Qualification_Secondary': 'secondary',
    'Qualification_PostGraduate': 'postgraduate',
    'Qualification_Professional': 'professional',
    'Qualification_MBA': 'mba',
    'Qualification_Vocational': 'vocational',
    'Qualification_PHD': 'phd'
})
df

,ID,TestStartTime,TestFinishTime,Gender,Qualification
0,992,2023-09-08 07:07:29.533,2023-09-08 07:09:26.197,male,unknown
1,1124,2023-09-27 16:03:59.047,2023-09-27 16:46:19.430,female,unknown
2,1158,2023-09-29 14:43:52.897,2023-09-29 15:04:57.667,male,bachelor
3,1285,2023-10-13 09:23:15.460,2023-10-13 09:52:39.973,male,unknown
4,1485,2023-10-23 07:47:29.690,2023-10-23 08:19:23.397,male,bachelor
...,...,...,...,...,...
160213,4982046,2024-02-19 08:34:39.187,2024-02-19 09:04:19.400,male,unknown
160214,4982097,2024-02-08 10:58:04.770,2024-02-08 11:35:20.650,male,unknown
160215,4982163,2024-02-22 12:34:33.257,2024-02-22 13:26:31.967,male,unknown
160216,4982214,2024-02-01 12:47:48.603,2024-02-01 13:20:29.923,male,unknown


In [12]:
df = df[df['TestFinishTime'] - df['TestStartTime'] != pd.Timedelta(hours=1)]
df

,ID,TestStartTime,TestFinishTime,Gender,Qualification
0,992,2023-09-08 07:07:29.533,2023-09-08 07:09:26.197,male,unknown
1,1124,2023-09-27 16:03:59.047,2023-09-27 16:46:19.430,female,unknown
2,1158,2023-09-29 14:43:52.897,2023-09-29 15:04:57.667,male,bachelor
3,1285,2023-10-13 09:23:15.460,2023-10-13 09:52:39.973,male,unknown
4,1485,2023-10-23 07:47:29.690,2023-10-23 08:19:23.397,male,bachelor
...,...,...,...,...,...
160213,4982046,2024-02-19 08:34:39.187,2024-02-19 09:04:19.400,male,unknown
160214,4982097,2024-02-08 10:58:04.770,2024-02-08 11:35:20.650,male,unknown
160215,4982163,2024-02-22 12:34:33.257,2024-02-22 13:26:31.967,male,unknown
160216,4982214,2024-02-01 12:47:48.603,2024-02-01 13:20:29.923,male,unknown


In [13]:
remaining_rows = df[(df['TestFinishTime'] - df['TestStartTime'] == pd.Timedelta(hours=1))]
if remaining_rows.empty:
    print("Er zijn geen rijen meer met een tijdsverschil van precies 1 uur.")
else:
    print("Er zijn nog steeds rijen met een tijdsverschil van precies 1 uur.")

Er zijn geen rijen meer met een tijdsverschil van precies 1 uur.


### Kolom toevoegen met hoelang ze er over gedaan hebben

In [14]:
df['test_duration'] = df['TestFinishTime'] - df['TestStartTime']
df['test_duration_seconds'] = df['test_duration'].dt.total_seconds()
df['test_duration_minutes'] = df['test_duration'].dt.total_seconds() / 60

C:\Users\Lorem\AppData\Local\Temp\ipykernel_18628\2198987834.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['test_duration'] = df['TestFinishTime'] - df['TestStartTime']
C:\Users\Lorem\AppData\Local\Temp\ipykernel_18628\2198987834.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['test_duration_seconds'] = df['test_duration'].dt.total_seconds()
C:\Users\Lorem\AppData\Local\Temp\ipykernel_18628\2198987834.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Data

In [15]:
df = df.drop(columns=['test_duration', 'TestStartTime', 'TestFinishTime'])
df

,ID,Gender,Qualification,test_duration_seconds,test_duration_minutes
0,992,male,unknown,116.664,1.944400
1,1124,female,unknown,2540.383,42.339717
2,1158,male,bachelor,1264.770,21.079500
3,1285,male,unknown,1764.513,29.408550
4,1485,male,bachelor,1913.707,31.895117
...,...,...,...,...,...
160213,4982046,male,unknown,1780.213,29.670217
160214,4982097,male,unknown,2235.880,37.264667
160215,4982163,male,unknown,3118.710,51.978500
160216,4982214,male,unknown,1961.320,32.688667


In [ ]:
fca_time = df.join(fca_df, [''])